# BioHub Cell Tracking During Development: a practical tutorial

This notebook explains the competition from the data model to a valid submission. It is written for data scientists who know basic NumPy and pandas but are new to biological cell tracking.

**Prerequisites:** Python, arrays, and basic optimization vocabulary. No cell-biology background is assumed.

By the end, you will be able to:

- interpret a 3D time-lapse movie and its tracking graph;
- explain why physical coordinates matter;
- detect synthetic nuclei with Difference-of-Gaussians (DoG);
- link detections with Hungarian assignment;
- construct the competition submission rows; and
- design validation that does not leak embryo identity.


## Outline

1. Understand the biological and graph prediction task
2. Inspect the competition files
3. Work in physical units
4. Build a small DoG + Hungarian tracker
5. Understand the official metric
6. Write valid submission rows
7. Validate safely and plan improvements


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import linear_sum_assignment

SEED = 42
rng = np.random.default_rng(SEED)

COMPETITION = 'biohub-cell-tracking-during-development'
KAGGLE_ROOTS = [
    Path(f'/kaggle/input/competitions/{COMPETITION}'),
    Path(f'/kaggle/input/{COMPETITION}'),
]
LOCAL_ROOT = Path('../data')
DATA_ROOT = next((p for p in KAGGLE_ROOTS if p.exists()), LOCAL_ROOT)
SCALE_UM = np.array([1.625, 0.40625, 0.40625])  # z, y, x

print('Data root:', DATA_ROOT)
print('Real data available:', DATA_ROOT.exists())


## 1. What are we predicting?

Each dataset is a movie with shape **(time, z, y, x)**. A bright nucleus is a proxy for one cell. For every timepoint we must predict cell-centre nodes, then add directed edges that say which cell at time `t` became which cell at `t+1`.

A normal continuation has one outgoing edge. A division is a fork: one parent node has two outgoing edges to daughter cells. The final output is therefore a **directed lineage graph**, not a segmentation mask.

```text
t=0          t=1          t=2
  A ---------- B ---------- C
               \
                ----------- D    (division)
```


In [ ]:
toy_nodes = pd.DataFrame({
    'node_id': [1, 2, 3, 4],
    't': [0, 1, 2, 2],
    'z': [10, 10, 10, 11],
    'y': [20, 21, 22, 20],
    'x': [30, 31, 32, 31],
})
toy_edges = pd.DataFrame({
    'source_id': [1, 2, 2],
    'target_id': [2, 3, 4],
})
display(toy_nodes)
display(toy_edges)


## 2. Competition data

Image movies are OME-Zarr directories ending in `.zarr`. Training labels are GEFF graph directories ending in `.geff`. Both are directory-based formats rather than single files. The following inventory cell works on Kaggle and remains harmless when the dataset is absent locally.


In [ ]:
def inventory(root: Path) -> pd.DataFrame:
    rows = []
    for split in ('train', 'test'):
        split_dir = root / split
        if not split_dir.exists():
            continue
        movies = sorted(split_dir.glob('*.zarr'))
        labels = sorted(split_dir.glob('*.geff'))
        rows.append({'split': split, 'movies': len(movies), 'graphs': len(labels)})
    return pd.DataFrame(rows, columns=['split', 'movies', 'graphs'])

inventory(DATA_ROOT)


In [ ]:
def read_movie_metadata(movie_path: Path) -> dict:
    metadata_path = movie_path / '0' / 'zarr.json'
    metadata = json.loads(metadata_path.read_text())
    return {
        'dataset': movie_path.stem,
        'shape_tzyx': tuple(metadata['shape']),
        'dtype': metadata['data_type'],
        'chunk_shape': tuple(
            metadata.get('chunk_grid', {}).get('configuration', {}).get('chunk_shape', [])
        ),
    }

train_movies = sorted((DATA_ROOT / 'train').glob('*.zarr')) if DATA_ROOT.exists() else []
if train_movies:
    display(pd.DataFrame(read_movie_metadata(p) for p in train_movies[:5]))
else:
    print('Attach the competition data on Kaggle to inspect real movie metadata.')


### Optional: view a real frame without importing Zarr

The competition stores one compressed image chunk per timepoint. This reader uses `blosc2` directly, avoiding the common offline-kernel failure `zarr is not installed`. If real data or Blosc2 is unavailable, the tutorial continues with synthetic data.


In [ ]:
def read_frame_without_zarr(movie_path: Path, t: int = 0) -> np.ndarray:
    import blosc2

    metadata = json.loads((movie_path / '0' / 'zarr.json').read_text())
    shape = tuple(metadata['shape'])
    dtype = np.dtype(metadata['data_type'])
    chunk_path = movie_path / '0' / 'c' / str(t) / '0' / '0' / '0'
    decoded = blosc2.decompress(chunk_path.read_bytes())
    return np.frombuffer(decoded, dtype=dtype).reshape(shape[1:]).copy()

real_frame = None
if train_movies:
    try:
        real_frame = read_frame_without_zarr(train_movies[0])
        plt.figure(figsize=(7, 7))
        plt.imshow(real_frame.max(axis=0), cmap='gray')
        plt.title(f'{train_movies[0].stem}: z-max projection, t=0')
        plt.axis('off')
        plt.show()
    except Exception as exc:
        print(f'Real-frame preview skipped: {type(exc).__name__}: {exc}')
else:
    print('Real-frame preview skipped: competition data is not attached.')


## 3. Why physical units matter

A voxel is anisotropic: one step in Z is `1.625 µm`, while one step in Y or X is `0.40625 µm`. Therefore a one-voxel Z error is physically four times larger than a one-voxel X/Y error. Detection suppression, matching, and linking gates should be defined in micrometres.

Downsampling X and Y by four gives an approximately isotropic working grid: `0.40625 × 4 = 1.625 µm`.


In [ ]:
def physical_distance(a_zyx, b_zyx, scale=SCALE_UM) -> float:
    return float(np.linalg.norm((np.asarray(a_zyx) - np.asarray(b_zyx)) * scale))

examples = pd.DataFrame([
    {'move': '+1 voxel in Z', 'distance_um': physical_distance((0, 0, 0), (1, 0, 0))},
    {'move': '+1 voxel in X', 'distance_um': physical_distance((0, 0, 0), (0, 0, 1))},
    {'move': '+4 voxels in X', 'distance_um': physical_distance((0, 0, 0), (0, 0, 4))},
])
examples


## 4. A minimal DoG + Hungarian tracker

We will generate a tiny 3D movie. The cells move slowly, and one cell divides in the final frame. Synthetic data makes every intermediate result visible and keeps this tutorial runnable without downloading the competition.


In [ ]:
def render_frame(shape, centres, sigma=(1.2, 2.5, 2.5), noise=0.03):
    impulse = np.zeros(shape, dtype=np.float32)
    for z, y, x in centres:
        impulse[int(z), int(y), int(x)] += 1.0
    image = gaussian_filter(impulse, sigma=sigma)
    image /= max(float(image.max()), 1e-8)
    image += rng.normal(0, noise, shape).astype(np.float32)
    return np.clip(image, 0, None)

true_centres = [
    [(5, 18, 18), (8, 44, 42)],
    [(5, 20, 20), (8, 42, 41)],
    [(5, 22, 22), (8, 39, 39), (8, 45, 43)],
]
movie = np.stack([render_frame((14, 64, 64), centres) for centres in true_centres])

fig, axes = plt.subplots(1, len(movie), figsize=(12, 4))
for t, (ax, frame) in enumerate(zip(axes, movie)):
    ax.imshow(frame.max(axis=0), cmap='gray')
    ax.set_title(f't={t}')
    ax.axis('off')
plt.tight_layout()


### Detect cells with Difference-of-Gaussians

DoG subtracts a broadly blurred image from a narrowly blurred image. Structures near the desired cell size remain bright while smooth background is removed. We then retain local maxima above a threshold. Production solutions use multiple scales, refine coordinates on the original volume, and perform physical non-maximum suppression.


In [ ]:
def detect_dog(frame, threshold=0.04, min_distance_vox=(2, 5, 5)):
    small = gaussian_filter(frame, sigma=(1.0, 2.0, 2.0))
    large = gaussian_filter(frame, sigma=(2.5, 5.0, 5.0))
    response = small - large
    footprint = tuple(2 * radius + 1 for radius in min_distance_vox)
    local_max = response == maximum_filter(response, size=footprint)
    coords = np.argwhere(local_max & (response >= threshold))
    return coords.astype(float), response

detections = []
responses = []
for frame in movie:
    coords, response = detect_dog(frame)
    detections.append(coords)
    responses.append(response)

[len(coords) for coords in detections]


In [ ]:
fig, axes = plt.subplots(1, len(movie), figsize=(12, 4))
for t, ax in enumerate(axes):
    ax.imshow(movie[t].max(axis=0), cmap='gray')
    if len(detections[t]):
        ax.scatter(detections[t][:, 2], detections[t][:, 1],
                   facecolors='none', edgecolors='tab:red', s=100)
    ax.set_title(f't={t}: {len(detections[t])} detections')
    ax.axis('off')
plt.tight_layout()


### What is a Hungarian tracker?

A Hungarian tracker repeatedly solves an **assignment problem** between two consecutive frames. Rows are cells detected at time `t`; columns are cells detected at time `t+1`. Each matrix entry is the cost of linking that pair—here, their distance in micrometres.

The Hungarian algorithm chooses a set of one-to-one links with the **smallest total cost**. It considers all links together, unlike a greedy tracker that assigns each old cell to its nearest available detection in sequence. This matters in crowded scenes: an apparently good early choice can leave a later cell with a very bad match.


For each adjacent pair of frames, the tracker performs four steps:

1. Convert `(z, y, x)` coordinates to physical micrometres.
2. Build the pairwise distance matrix $C_{ij}=\lVert p_i^{(t)}-p_j^{(t+1)}\rVert_2$.
3. Solve $\min \sum C_{ij}x_{ij}$ subject to each detection being used at most once.
4. Reject assignments beyond a movement gate, such as `8 µm`.

Repeating this for every adjacent frame turns pairwise assignments into tracks: each accepted match becomes a directed graph edge.


In [ ]:
# A one-dimensional example in micrometres.
old_positions = np.array([0.0, 1.1])       # cells A and B at time t
new_positions = np.array([1.0, -2.0])      # detections 1 and 2 at time t+1
cost = np.abs(old_positions[:, None] - new_positions[None, :])
cost_table = pd.DataFrame(cost, index=['old A', 'old B'], columns=['new 1', 'new 2'])
display(cost_table)

# Row-order greedy: A takes its nearest detection, then B takes what remains.
greedy_pairs = [('old A', 'new 1', cost[0, 0]), ('old B', 'new 2', cost[1, 1])]

# Hungarian: choose both links jointly to minimize their total cost.
row_idx, col_idx = linear_sum_assignment(cost)
hungarian_pairs = [
    (cost_table.index[i], cost_table.columns[j], cost[i, j])
    for i, j in zip(row_idx, col_idx)
]

pd.DataFrame({
    'method': ['row-order greedy', 'Hungarian'],
    'links': [[(a, b) for a, b, _ in greedy_pairs],
              [(a, b) for a, b, _ in hungarian_pairs]],
    'total_cost_um': [sum(x[2] for x in greedy_pairs),
                      sum(x[2] for x in hungarian_pairs)],
})


The greedy order produces links with total cost `4.1 µm`; Hungarian assignment finds the globally better combination at `2.1 µm`. The algorithm is therefore global **within one frame pair**, although the complete tracker is still local in time.

For the synthetic movie, the next cell uses physical 3D distances. Costs above the gate receive a very large value before optimization, then are omitted from the graph. This lets cells appear or disappear instead of forcing every detection to match.


In [ ]:
def link_adjacent_frames(frames, max_link_um=8.0, scale=SCALE_UM):
    node_rows, edges, frame_ids = [], [], []
    next_id = 1

    for t, coords in enumerate(frames):
        ids = []
        for z, y, x in coords:
            node_rows.append({'node_id': next_id, 't': t, 'z': z, 'y': y, 'x': x})
            ids.append(next_id)
            next_id += 1
        frame_ids.append(ids)

    for t in range(len(frames) - 1):
        a, b = frames[t], frames[t + 1]
        if not len(a) or not len(b):
            continue
        distances = np.linalg.norm(
            a[:, None, :] * scale - b[None, :, :] * scale, axis=2
        )
        blocked_cost = max_link_um * 1000.0 + 1.0
        gated_cost = np.where(distances <= max_link_um, distances, blocked_cost)
        rows, cols = linear_sum_assignment(gated_cost)
        for old, new in zip(rows, cols):
            if distances[old, new] <= max_link_um:
                edges.append({
                    'source_id': frame_ids[t][old],
                    'target_id': frame_ids[t + 1][new],
                    'distance_um': distances[old, new],
                })
    return pd.DataFrame(node_rows), pd.DataFrame(edges)

pred_nodes, pred_edges = link_adjacent_frames(detections)
display(pred_nodes.round(2))
display(pred_edges.round(2))


#### What Hungarian assignment does not solve

- **Division:** one-to-one matching cannot connect one parent to two daughters. Add a separate, conservative division rule or global lineage model.
- **Missed detections:** adjacent-frame matching cannot cross a missing frame. Insert an interpolated node and two consecutive edges when closing a one-frame gap.
- **Motion ambiguity:** nearest distance ignores velocity and appearance. A motion-aware cost can compare detections with predicted positions.
- **Global lineage consistency:** solving each frame pair independently can make locally cheap but globally inconsistent choices. ILP or learned global trackers address this at higher complexity.

So “Hungarian tracker” describes the assignment engine, not a complete biological lineage model. It is valuable here because it is fast, auditable, and a strong baseline.


## 5. How scoring works

The official score is:

$$\text{adjusted edge Jaccard} + 0.1 \times \text{division Jaccard}$$

A predicted edge is correct only when **both endpoints** match annotated nodes within the physical matching gate and the corresponding ground-truth edge exists. Detection quality therefore affects edge quality twice. Excess total node count is also penalized.

Division scoring is tolerant in time and lineage connectivity; it is not simply exact parent-ID matching. Use the organizer's evaluator for model selection. The function below demonstrates ordinary set Jaccard only to build intuition—it is **not** a replacement for the official metric.


In [ ]:
def set_jaccard(predicted, truth):
    predicted, truth = set(predicted), set(truth)
    intersection = len(predicted & truth)
    union = len(predicted | truth)
    return intersection / union if union else 1.0

truth_edges = {(1, 2), (2, 3), (2, 4)}
example_predictions = {(1, 2), (2, 3), (3, 4)}
set_jaccard(example_predictions, truth_edges)


### Metric-critical pitfalls

- Do not optimize node F1 as a substitute for adjusted edge Jaccard.
- Do not weight divisions equally with edges; their final weight is only `0.1`.
- Do not bridge a missing frame with a direct `t → t+2` edge. Insert a node at `t+1` and create two consecutive edges.
- Do not compute distances in raw voxel coordinates.
- Do not install Zarr wheels into the live offline kernel if they replace NumPy/SciPy. Keep the official evaluator isolated.


## 6. Submission format

The CSV has one row per node and one row per edge. Node rows use `node_id, t, z, y, x`; edge rows use `source_id, target_id`. Fields that do not apply are set to `-1`. Node IDs must be unique within each dataset.


In [ ]:
SUBMISSION_COLUMNS = [
    'id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
    'source_id', 'target_id',
]

def graph_to_submission_rows(dataset, nodes, edges):
    rows = []
    for row in nodes.itertuples(index=False):
        rows.append({
            'dataset': dataset, 'row_type': 'node',
            'node_id': int(row.node_id), 't': int(row.t),
            'z': int(round(row.z)), 'y': int(round(row.y)), 'x': int(round(row.x)),
            'source_id': -1, 'target_id': -1,
        })
    for row in edges.itertuples(index=False):
        rows.append({
            'dataset': dataset, 'row_type': 'edge',
            'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
            'source_id': int(row.source_id), 'target_id': int(row.target_id),
        })
    result = pd.DataFrame(rows)
    result.insert(0, 'id', np.arange(len(result), dtype=int))
    return result[SUBMISSION_COLUMNS]

tutorial_submission = graph_to_submission_rows('tutorial_movie', pred_nodes, pred_edges)
tutorial_submission


In [ ]:
def validate_submission_table(table):
    assert list(table.columns) == SUBMISSION_COLUMNS
    assert table['id'].is_unique
    assert set(table['row_type']).issubset({'node', 'edge'})
    node_rows = table['row_type'].eq('node')
    edge_rows = table['row_type'].eq('edge')
    assert (table.loc[node_rows, ['source_id', 'target_id']] == -1).all().all()
    assert (table.loc[edge_rows, ['node_id', 't', 'z', 'y', 'x']] == -1).all().all()
    return {'rows': len(table), 'nodes': int(node_rows.sum()), 'edges': int(edge_rows.sum())}

validate_submission_table(tutorial_submission)


## 7. Validation without leakage

Dataset names group fields of view by embryo prefix. Appearance and development stage can be embryo-specific, so splitting frames or fields randomly can leak identity. Hold out complete embryos, run the entire tracking pipeline, and score with the exact evaluator.


In [ ]:
def embryo_id(dataset_name: str) -> str:
    return dataset_name.split('_')[0]

example_datasets = [
    'embryoA_fov1', 'embryoA_fov2',
    'embryoB_fov1', 'embryoC_fov1',
]
validation_embryo = 'embryoC'
split_table = pd.DataFrame({
    'dataset': example_datasets,
    'embryo': [embryo_id(name) for name in example_datasets],
})
split_table['split'] = np.where(
    split_table['embryo'].eq(validation_embryo), 'validation', 'train'
)
split_table


## Exercise: detection density versus graph quality

Run the detector with thresholds `0.03`, `0.04`, and `0.06`. Before running the next cell, predict which threshold produces the most nodes. Then inspect whether every additional node is useful.

This is the central tradeoff in the competition: lower thresholds can recover dim cells, but excess nodes and incorrect links can reduce the adjusted edge score.


In [ ]:
def threshold_experiment(thresholds=(0.03, 0.04, 0.06)):
    rows = []
    for threshold in thresholds:
        detected = [detect_dog(frame, threshold=threshold)[0] for frame in movie]
        nodes, edges = link_adjacent_frames(detected)
        rows.append({
            'threshold': threshold,
            'nodes': len(nodes),
            'edges': len(edges),
            'detections_by_frame': [len(x) for x in detected],
        })
    return pd.DataFrame(rows)

threshold_experiment()


## Practical strategy

A disciplined progression is:

1. **Exact evaluator:** run the organizer metric in an isolated offline environment.
2. **Auditable baseline:** multi-scale DoG → physical NMS → centroid refinement → Hungarian linking → one-frame interpolation → isolated-node pruning.
3. **Detector sweep:** tune threshold/density and NMS radius while keeping linking fixed.
4. **Linker ablation:** tune the link gate, then test velocity-aware or two-pass linking.
5. **Conservative divisions:** add only after edge quality is stable because division weight is `0.1`.
6. **Ceiling raiser:** separately evaluate the artifact-backed 3D U-Net + learned edge model + ILP tracker.

Change one family of parameters at a time and record node count, node recall, edge TP/FP/FN, adjusted edge Jaccard, division counts, and runtime for every held-out embryo.


## Takeaways

- The target is a lineage graph built from 3D detections and temporal edges.
- Anisotropic voxel spacing makes physical units mandatory.
- Detection recall, node-count control, and temporal linking are coupled through edge scoring.
- Hungarian assignment is a transparent baseline; divisions and gaps require explicit logic.
- Exact embryo-disjoint validation is more trustworthy than notebook titles, proxy metrics, or a single leaderboard result.

For a complete competition-ready implementation, continue with `biohub-exact-dog-hungarian-baseline.ipynb` in this workspace.
